In [4]:
import pandas as pd
import dask.dataframe as dd
import json

#Create CSV file that named subID.csv to input all the subsscriptions uner the 'SubID' column and save it to the same folder.
subList=pd.read_csv('sub_llist.csv')['SubID'].values.tolist()

#Input your downloaded usage CSV format file name here
url = 'DetailedUsageCsv.February.09.01.59.108170.csv' 

filteredList = []
for a in subList:
    if a not in filteredList:
        filteredList.append(a)

usage = dd.read_csv(url, skiprows=2,
                    dtype={'ServiceInfo2': 'object'
                          })
filtered = usage.loc[usage["SubscriptionGuid"].isin(filteredList)]
pd_usage = filtered.compute()

## GET RI orderID 
reservation=pd_usage[pd_usage['AdditionalInfo'].str.contains('"ReservationOrderId"', na=False)]
RiList=reservation.loc[:,'AdditionalInfo'].values.tolist()
ROList =[]

for list in RiList:
    roid = json.loads(list)["ReservationOrderId"]
    if roid not in ROList:
        ROList.append(roid)
        
##Reservation Order ID
if(len(ROList) == 0):
    print("##### NO RI #####")
else:
    print("RESERVATION ORDER ID LIST")
    print(ROList)
        
## USAGE
pd.options.display.float_format = '{:.7f}'.format
summary = pd.pivot_table(pd_usage, index=["SubscriptionGuid"], values=["ExtendedCost"], aggfunc="sum", margins=True, margins_name="Total")
summary.reset_index(inplace=True)
TotalAmount = summary.tail(n=1)['ExtendedCost']
refund = TotalAmount*0.1
summary.loc[len(summary.index)] = ['Refund(Total*10%)',refund.values[0]]
display(summary)

##### NO RI #####


,SubscriptionGuid,ExtendedCost
0,6481f356-56fd-415f-bb1e-4788ba8a1955,35976667.5696736
1,Total,35976667.5696736
2,Refund(Total*10%),3597666.7569674


In [6]:
##Download
file_name = 'refund.xlsx'
with pd.ExcelWriter(file_name) as writer:
    summary.to_excel(writer, sheet_name="refund", startrow=1, startcol=1, index=False)
    #filtered.to_excel(writer, sheet_name="usage", startrow=0, startcol=0, index=False)
display("DONE!! " + file_name + " file")

'DONE!! refund.xlsx file'